# Registration rejects a non-callable finalizer

A present value is not enough: the adapter must be able to call finalize.

Use the repository Python environment and run the cells in order. This notebook has its own setup; no previous notebook is required.

## 1. Import the implementation

The lifecycle code comes from AOA itself. The example supplies only the resource or external connection used to observe it.

In [1]:
import asyncio

from aoa.action_machine.graph.core.exclude_graph_model import exclude_graph_model
from aoa.action_machine.resources.base_resource import BaseResource
from aoa.action_machine.resources.connection_scope import connection_scope
from aoa.action_machine.resources.per_call_connection import PerCallConnection

## 2. Define the objects used in this case

These definitions give the case an observable result without introducing unrelated business behavior.

In [2]:
@exclude_graph_model
class TraceResource(BaseResource):
    """Record lifecycle calls without opening an external connection."""
    def __init__(self, events):
        """Keep the event list supplied by the example."""
        self.events = events
    def get_wrapper_class(self):
        """Allow direct use by nested actions."""
        return None
    async def open(self):
        """Record acquisition."""
        self.events.append("open")
    async def close(self):
        """Record release."""
        self.events.append("close")

from aoa.action_machine.resources.per_call_connection import validate_connection_entries

## 4. Run and check the observation

The assertions test this case’s stated behavior. Printed output is recorded below; expected failures are caught explicitly.

In [3]:
try:
    validate_connection_entries({"db": PerCallConnection(lambda: TraceResource([]), finalize=None)})
except TypeError as error:
    assert str(error) == "PerCallConnection.finalize must be callable."
    print(str(error))
else:
    raise AssertionError("Non-callable finalize was accepted")

PerCallConnection.finalize must be callable.


## 5. Explain the result

The route-record validation uses this check. It checks callability without running the factory or finalizer.